In [3]:
import json
import sys

sys.stdin.reconfigure(encoding="utf-8")

from db import query_db
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage

# 1. DB_SCHEMA.txt 파일을 읽어서 DB_SCHEMA 변수에 저장합니다.
with open("DB_SCHEMA.txt", "r", encoding="utf-8") as f:
    DB_SCHEMA = f.read()

# client = OpenAI(api_key="sk-")
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL      = "https://openrouter.ai/api/v1"

ELICE_API_KEY       = os.getenv("ELICE_API_KEY")
ELICE_URL           = os.getenv("ELICE_URL")

# MODEL_NAME          = 'nvidia/nemotron-3-super-120b-a12b:free'
MODEL_NAME          = "zai-org/glm-4.7"

# client = OpenAI(
#     api_key  = OPENROUTER_API_KEY, # 여기에 실제 API 키를 넣으세요
#     base_url = OPENROUTER_URL,
# )

client = OpenAI(
    api_key  = ELICE_API_KEY, # 여기에 실제 API 키를 넣으세요
    base_url = ELICE_URL,
)


def get_response(chat_log: list, tools: dict = None) -> ChatCompletionMessage:
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=chat_log, tools=tools
    )
    if response.choices[0].message.tool_calls:
        tool_call_results = []
        for tool_call in response.choices[0].message.tool_calls:
            call_id = tool_call.id

            if tool_call.function.name == "query_db":
                keywords = json.loads(tool_call.function.arguments)["query"]
                result = query_db(keywords)

                call_result_message = {
                    "tool_call_id": call_id,
                    "role": "tool",
                    "content": json.dumps(
                        {"result": str(result)},
                        ensure_ascii=False,
                    ),
                }
                tool_call_results.append(call_result_message)

        messages = chat_log + [response.choices[0].message] + tool_call_results
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )

    return response.choices[0].message


query_db_info = {
    "name": "query_db",
    "description": "데이터베이스에 쿼리를 실행하고 결과를 반환합니다."
    + DB_SCHEMA,  # DB 스키마 정보 추가
    "parameters": {
        "type": "object",
        "properties": {None},  # 2. query의 type과 description을 추가합니다.
        "required": ["query"],
        "additionalProperties": False,
    },
}


def main():
    tools = [
        {
            "type": "function",
            "function": query_db_info,
        }
    ]
    chat_log = [
        {
            "role": "system",
            "content": "너는 유용한 고객 지원 어시스턴트입니다."
            + "제공된 tools를 사용하여 사용자를 도와줘."
            + "함수를 호출했다면 그 데이터를 기반으로 답변해야 해.",
        },
    ]

    for i in range(2):
        chat_log.append(
            {
                "role": "user",
                "content": input("User> "),  # 유저 입력
            }
        )
        response = get_response(chat_log, tools=tools)
        print("AI>", response.content)
        chat_log.append(response)


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'session'

In [2]:
!pip install session

ERROR: Could not find a version that satisfies the requirement session (from versions: none)
ERROR: No matching distribution found for session

[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: C:\Users\rkfka\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip
